# Log-time finite-size analysis (one control value, several L)

Reads data from `../generators/get_slidding_p_time_log_fss.jl` or
`../generators/get_upper_lower_binary_time_log_fss.jl`.

* **Activity vs $t$ for each $L$**, with $t=L$ marked. For $t<L$ every $L$ must agree
  exactly (light cone), which also checks the data.
* **Two collapses of $A(t,L)$ against $t/L^z$:**
  - power-law form $A\,t^{\delta}$ with $(z,\delta)$ = `Z_GUESS`, `DELTA_GUESS`;
  - BVH form $A\,(\ln t)^{\bar\delta}$ with $z=1$ and $\bar\delta$ = `ALPHA_GUESS`.

  BVH's $z=1$ carries $(\ln t)^{-y}$ corrections, so expect some residual drift.
* **Lifetimes.** Each sample is one disorder realization. Its absorption time comes from
  the log grid, and the median lifetime $\tau(L)$ is plotted against $L$ on log–log axes.
  A slope of $z_{\rm eff}=1/\kappa$ that does not settle, or that moves with the control
  value, is the temporal-Griffiths behaviour BVH predict (review 3.1).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))          # run from stavskya_mc/block_disorder/analysis
import time_log_tools as tl

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.2, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 13})

In [ ]:
# ---- parameters (mirror the generator you ran) ------------------------------
MODEL = "slidding_p"            # "slidding_p" or "window_binary"
DATA_ROOT = "../../data/time_log"
L_VALS = [1250, 2500, 5000, 10000, 20000]
BLOCK_LEN = 1
TIME_PREFACT = 4500.0
PPD = 20
N_SAMPLES = 2000
OFFSET = 0
# sliding p
UPPER_VAL = 0.43
LOWER_DIV = 20
P_C = 0.47882
# upper/lower binary (lower_div = LOWER_DIV)
AVG_EPS_C = 0.27033
P_VAL = 0.8
# power-law collapse
Z_GUESS = 1.45
DELTA_GUESS = 0.094
ALPHA_GUESS = 1.0                                       # BVH collapse (z = 1)
FIG_DIR = "figs"


In [ ]:
if MODEL == "slidding_p":
    MODEL_DIR = "time_rand_slidding_p"
    u, l, pv = UPPER_VAL, round(UPPER_VAL / LOWER_DIV, 6), P_C
    label = f"p = {P_C}"
else:
    MODEL_DIR = "time_rand_window_binary"
    f = P_VAL + (1 - P_VAL) / LOWER_DIV
    u = round(AVG_EPS_C / f, 6); l = round(u / LOWER_DIV, 6); pv = P_VAL
    label = f"eps_bar = {round(P_VAL * u + (1 - P_VAL) * l, 6)}"
Path(FIG_DIR).mkdir(exist_ok=True)
runs = {L: tl.load_time_log_run(DATA_ROOT, MODEL_DIR, L, u, l, pv, BLOCK_LEN, TIME_PREFACT, PPD, N_SAMPLES, offset=OFFSET)
        for L in L_VALS}
cmap = plt.colormaps["Blues"].resampled(len(L_VALS) + 2)
colors = {L: cmap(i + 2) for i, L in enumerate(L_VALS)}
pd.DataFrame({L: {"samples": r.n, "t_max": r.times[-1], "surviving at t_max": r.surviving_fraction[-1]}
              for L, r in runs.items()}).T

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for L, r in runs.items():
    m = (r.times > 0) & (r.mean > 0)
    ax.plot(np.log10(r.times[m]), np.log10(r.mean[m]), color=colors[L], label=f"L={L}")
    ax.axvline(np.log10(L), color=colors[L], ls=":", lw=1)
ax.set_xlabel(r"$\log_{10} t$"); ax.set_ylabel(r"$\log_{10} A$"); ax.set_title(f"{MODEL}, {label}, block_len={BLOCK_LEN}")
ax.legend(); fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_fss_bl{BLOCK_LEN}_activity.png", dpi=150)

# light-cone check: for t < min(L) all sizes must agree within errors
# (the log grids differ between L because t_max = L * TIME_PREFACT, so compare on common times)
Lmin = min(L_VALS); ref = runs[max(L_VALS)]
for L, r in runs.items():
    common, i_r, i_ref = np.intersect1d(r.times, ref.times, return_indices=True)
    m = (common > 0) & (common < Lmin)
    if m.any() and L != max(L_VALS):
        z = (r.mean[i_r][m] - ref.mean[i_ref][m]) / np.hypot(r.sem[i_r][m], ref.sem[i_ref][m])
        print(f"L={L}: max |difference|/SE vs L={max(L_VALS)} over {m.sum()} common times t < {Lmin}: {np.nanmax(np.abs(z)):.2f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for L, r in runs.items():
    m = (r.times > 0) & (r.mean > 0)
    tt, A = r.times[m], r.mean[m]
    axs[0].plot(np.log10(tt / L ** Z_GUESS), np.log10(A * tt ** DELTA_GUESS), color=colors[L], label=f"L={L}")
    m2 = tt > np.e
    axs[1].plot(np.log10(tt[m2] / L), np.log10(A[m2] * np.log(tt[m2]) ** ALPHA_GUESS), color=colors[L])
axs[0].set_xlabel(r"$\log_{10}(t/L^{z})$"); axs[0].set_ylabel(r"$\log_{10}(A\,t^{\delta})$")
axs[0].set_title(f"power law: z={Z_GUESS}, delta={DELTA_GUESS}")
axs[1].set_xlabel(r"$\log_{10}(t/L)$"); axs[1].set_ylabel(r"$\log_{10}(A\,(\ln t)^{\bar\delta})$")
axs[1].set_title(f"BVH: z=1, delta_bar={ALPHA_GUESS}")
axs[0].legend(); fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_fss_bl{BLOCK_LEN}_collapses.png", dpi=150)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
tau = {}
for L, r in runs.items():
    axs[0].plot(np.log10(r.times[r.times > 0] / L), r.surviving_fraction[r.times > 0], color=colors[L], label=f"L={L}")
    at = tl.absorption_times(r)
    tau[L] = {"median lifetime": np.median(at),                   # inf if fewer than half were absorbed
              "absorbed fraction": np.isfinite(at).mean()}
tau = pd.DataFrame(tau).T
axs[0].set_xlabel(r"$\log_{10}(t/L)$"); axs[0].set_ylabel("surviving fraction"); axs[0].legend()
ok = tau[np.isfinite(tau["median lifetime"])]
if len(ok) >= 2:
    x, y = np.log10(ok.index.to_numpy(float)), np.log10(ok["median lifetime"].to_numpy(float))
    slope = np.polyfit(x, y, 1)[0]
    axs[1].plot(x, y, "o-"); axs[1].set_title(f"slope z_eff = 1/kappa = {slope:.3f}")
    print(f"median-lifetime slope z_eff = {slope:.3f}  (BVH at criticality: 1, clean DP: 1.58)")
else:
    axs[1].set_title("fewer than two L with a finite median lifetime: run longer")
axs[1].set_xlabel(r"$\log_{10} L$"); axs[1].set_ylabel(r"$\log_{10}\tau_{\rm median}$")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{MODEL}_fss_bl{BLOCK_LEN}_lifetimes.png", dpi=150)
tau